In [1]:
import numpy as np

In [3]:
t1 = np.array([-1.5, -0.8, 0.0, 0.9, 2.3], dtype=np.float32)
t2 = np.array([0.1, 0.5, 1.2, 2.0, 3.5], dtype=np.float32)

t3 = np.array([-3.0, -2.1, -1.4, -0.6, -0.1], dtype=np.float32)

t4 = np.array([5.0, 5.0, 5.0], dtype=np.float32)

t5 = np.array([1e-9, 2e-9, -1e-9], dtype=np.float32)

tensors = {
    "Tensor 1": t1,
    "Tensor 2": t2,
    "Tensor 3": t3,
    "Tensor 4": t4,
    "Tensor 5": t5
}

In [5]:
def calculate_scale_zero_point(tensor, q_min=-128, q_max=127):
    x_min = np.min(tensor)
    x_max = np.max(tensor)

    if np.isclose(x_min, x_max):
        scale = 1.0
        zero_point = 0
        return scale, zero_point

    scale = (x_max - x_min) / (q_max - q_min)

    if scale < 1e-12:
        scale = 1e-12

    zero_point = np.round(q_min - (x_min / scale))

    zero_point = np.clip(zero_point, q_min, q_max)

    return float(scale), int(zero_point)

In [7]:
def quantize_tensor(tensor, scale, zero_point):
    quantized = np.round(tensor / scale) + zero_point
    quantized = np.clip(quantized, -128, 127)
    return quantized.astype(np.int8)

In [9]:
def dequantize_tensor(quantized_tensor, scale, zero_point):
    return (quantized_tensor.astype(np.float32) - zero_point) * scale

In [11]:
for name, tensor in tensors.items():

    scale, zero_point = calculate_scale_zero_point(tensor)

    quantized = quantize_tensor(tensor, scale, zero_point)

    dequantized = dequantize_tensor(quantized, scale, zero_point)

    mae = np.mean(np.abs(tensor - dequantized))

    print("=" * 50)
    print(name)
    print("=" * 50)

    print("Tensor values:")
    print(tensor)

    print("\nTensor min:")
    print(np.min(tensor))

    print("\nTensor max:")
    print(np.max(tensor))

    print("\nScale:")
    print(scale)

    print("\nZero Point:")
    print(zero_point)

    print("\nQuantized Tensor:")
    print(quantized)

    print("\nDequantized Tensor:")
    print(dequantized)

    print("\nMean Absolute Error:")
    print(mae)

    print("\n")

Tensor 1
Tensor values:
[-1.5 -0.8  0.   0.9  2.3]

Tensor min:
-1.5

Tensor max:
2.3

Scale:
0.014901960597318761

Zero Point:
-27

Quantized Tensor:
[-128  -81  -27   33  127]

Dequantized Tensor:
[-1.505098   -0.80470586  0.          0.8941176   2.2949018 ]

Mean Absolute Error:
0.004156864


Tensor 2
Tensor values:
[0.1 0.5 1.2 2.  3.5]

Tensor min:
0.1

Tensor max:
3.5

Scale:
0.013333333707323262

Zero Point:
-128

Quantized Tensor:
[-120  -90  -38   22  127]

Dequantized Tensor:
[0.10666667 0.50666666 1.2        2.         3.4       ]

Mean Absolute Error:
0.022666646


Tensor 3
Tensor values:
[-3.  -2.1 -1.4 -0.6 -0.1]

Tensor min:
-3.0

Tensor max:
-0.1

Scale:
0.011372549393597772

Zero Point:
127

Quantized Tensor:
[-128  -58    4   74  118]

Dequantized Tensor:
[-2.9        -2.1039217  -1.3988236  -0.6027451  -0.10235295]

Mean Absolute Error:
0.022039209


Tensor 4
Tensor values:
[5. 5. 5.]

Tensor min:
5.0

Tensor max:
5.0

Scale:
1.0

Zero Point:
0

Quantized Tensor:
[5 

# Observations

## Tensor 1 – Mixed Positive and Negative Values

- Since the tensor contains both positive and negative values, the calculated zero point lies near the center of the INT8 range.
- The reconstructed tensor is very close to the original, resulting in a low Mean Absolute Error (MAE).

---

## Tensor 2 – All Positive Values

- The tensor contains only positive values.
- The zero point shifts toward the lower end of the INT8 range so that the positive values can be represented efficiently.
- Small reconstruction errors occur because of rounding during quantization.

---

## Tensor 3 – All Negative Values

- Since all values are negative, the zero point shifts toward the upper end of the INT8 range.
- This allows the available quantization levels to represent the negative values with good precision.
- The dequantized values remain close to the originals.

---

## Tensor 4 – Constant Tensor

- All tensor elements have the same value.
- The normal scale formula would divide by zero because the range is zero.
- This edge case is handled by assigning a default scale of **1.0** and a zero point of **0**.
- The tensor can still be quantized and dequantized safely without runtime errors.

---

## Tensor 5 – Very Small Floating-Point Values

- The tensor values are extremely close to zero.
- The calculated scale is very small, allowing these tiny values to be represented without collapsing them all to the same quantized value.
- Using a minimum threshold for the scale prevents numerical instability.

---

## Overall Conclusion

- The scale determines the spacing between quantized levels.
- The zero point shifts the floating-point range into the INT8 range.
- Proper handling of edge cases such as constant tensors and very small ranges is essential for robust quantization.
- Affine quantization preserves the original tensor well while significantly reducing numerical precision.